<a href="https://colab.research.google.com/github/Taswell11111/ParcelNinja/blob/main/CSV_XLS_to_JSON_Merger.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import json
import os
import sys
import glob
import tkinter as tk
from tkinter import filedialog, messagebox

try:
    import pandas as pd
except ImportError:
    print("Error: 'pandas' library is missing.")
    print("Please install required libraries by running: pip install pandas openpyxl xlrd")
    exit()

def load_file(file_path):
    try:
        df = pd.read_excel(file_path)
        return df.fillna('')
    except Exception:
        pass

    try:
        df = pd.read_csv(file_path, sep=None, engine='python', encoding='utf-8-sig')
        return df.fillna('')
    except Exception as e:
        print(f"Failed to read {os.path.basename(file_path)}: {e}")
        return None

def merge_files_to_json(file_paths, output_file='Bounty_Recon_Merged.json'):
    merged_data = {}

    for file_path in file_paths:
        if not os.path.exists(file_path):
            continue

        print(f"Processing: {os.path.basename(file_path)}...")
        df = load_file(file_path)

        if df is None or df.empty:
            continue

        columns = [str(col).strip() for col in df.columns]
        df.columns = columns
        records = df.to_dict('records')

        for row in records:
            clean_row = {str(k).strip(): str(v).strip() for k, v in row.items()}

            # Find the Order Name (Handles Seller Portal, Shopify, and Returns)
            order_name = clean_row.get('Source Store Order ID') or clean_row.get('Name')
            if not order_name:
                continue

            if order_name not in merged_data:
                merged_data[order_name] = {"OrderName": order_name, "items": [], "returns": []}

            data = merged_data[order_name]

            # Ensure returns list is initialized
            if "returns" not in data:
                data["returns"] = []

            # --- Detect if this is a RETURN record ---
            if clean_row.get('Return ID'):
                ret_id = clean_row.get('Return ID')

                # Check if we already created a record for this specific Return ID
                existing_ret = next((r for r in data["returns"] if r["ReturnID"] == ret_id), None)
                if not existing_ret:
                    existing_ret = {
                        "ReturnID": ret_id,
                        "SourceShipmentID": clean_row.get('Source Shipment ID', ''),
                        "ReturnDate": clean_row.get('Return Date', ''),
                        "Courier": clean_row.get('Courier', ''),
                        "TrackingNo": clean_row.get('Tracking No', ''),
                        "Status": clean_row.get('Status', ''),
                        "StatusDate": clean_row.get('Status Date', ''),
                        "items": []
                    }
                    data["returns"].append(existing_ret)

                # Add returned item details to this return
                item_name = clean_row.get('Item Name')
                quantity = clean_row.get('Quantity')
                if item_name:
                    existing_ret["items"].append({
                        "name": item_name,
                        "quantity": quantity if quantity else "1.0"
                    })

                continue # Skip processing the rest of the standard shipment logic for this row

            # --- Shopify Specific Data ---
            if clean_row.get('Order Fulfillment Status'):
                data['ShopifyStatus'] = clean_row.get('Order Fulfillment Status')
            if clean_row.get('Created At'):
                data['ShopifyCreatedAt'] = clean_row.get('Created At')

            track_no_shop = clean_row.get('Fulfillment: Tracking Number')
            track_co_shop = clean_row.get('Fulfillment: Tracking Company')
            if track_no_shop and track_no_shop != '-':
                data["TrackingNumber"] = track_no_shop
            if track_co_shop and track_co_shop != 'To be determined':
                data["TrackingCompany"] = track_co_shop

            # --- Seller Portal Data ---
            if clean_row.get('Shipment ID'):
                data['ShipmentID'] = clean_row.get('Shipment ID')

            store_code = clean_row.get('Source Store', '')
            if store_code:
                store_map = {"CL": "Diesel", "SD": "Superdry"}
                data['Store'] = store_map.get(store_code, store_code)

            if clean_row.get('Status'):
                data['SellerPortalStatus'] = clean_row.get('Status')
            if clean_row.get('Status Date'):
                data['SellerPortalStatusDate'] = clean_row.get('Status Date')
            if clean_row.get('Order Date'):
                data['OrderDate'] = clean_row.get('Order Date')
            if clean_row.get('Customer Name'):
                data['CustomerName'] = clean_row.get('Customer Name')
            if clean_row.get('Address Line 1'):
                data['AddressLine1'] = clean_row.get('Address Line 1')
            if clean_row.get('Address Line 2'):
                data['AddressLine2'] = clean_row.get('Address Line 2')
            if clean_row.get('City'):
                data['City'] = clean_row.get('City')
            if clean_row.get('Region'):
                data['Region'] = clean_row.get('Region')
            if clean_row.get('State'):
                data['State'] = clean_row.get('State')

            track_no_sp = clean_row.get('Tracking No')
            track_co_sp = clean_row.get('Courier')
            if track_no_sp:
                data["TrackingNumber"] = track_no_sp
            if track_co_sp:
                data["TrackingCompany"] = track_co_sp

            # Add outbound shipment items
            item_name = clean_row.get('Item Name')
            quantity = clean_row.get('Quantity')
            if item_name:
                data["items"].append({
                    "name": item_name,
                    "quantity": quantity if quantity else "1.0"
                })

    # Format the dictionary into a list
    final_list = []
    for order_name, data in merged_data.items():
        data.setdefault("ShopifyStatus", "Unknown")
        data.setdefault("SellerPortalStatus", "Unknown")
        data.setdefault("TrackingNumber", "-")
        data.setdefault("TrackingCompany", "To be determined")
        data.setdefault("ShopifyCreatedAt", "")
        data.setdefault("SellerPortalStatusDate", "")
        data.setdefault("ShipmentID", "")
        data.setdefault("Store", "")
        data.setdefault("OrderDate", "")
        data.setdefault("CustomerName", "N/A")
        data.setdefault("AddressLine1", "")
        data.setdefault("AddressLine2", "")
        data.setdefault("City", "")
        data.setdefault("Region", "")
        data.setdefault("State", "")
        final_list.append(data)

    try:
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(final_list, f, indent=2)

        success_msg = f"Successfully extracted data.\nTotal unique orders processed: {len(final_list)}\nSaved as: {output_file}"
        print(success_msg)
        return success_msg
    except Exception as e:
        error_msg = f"Failed to save JSON file: {str(e)}"
        print(error_msg)
        return error_msg

if __name__ == '__main__':
    file_paths = []

    if 'google.colab' in sys.modules:
        from google.colab import files
        print("Google Colab detected. Please upload your XLSX or CSV files below:")
        uploaded = files.upload()
        file_paths = list(uploaded.keys())
    elif len(sys.argv) > 1 and not sys.argv[1].startswith('-') and not 'ipykernel' in sys.argv[0]:
        file_paths = sys.argv[1:]
    else:
        try:
            root = tk.Tk()
            root.withdraw()
            selected_files = filedialog.askopenfilenames(
                title="Select Export files (CSV, XLS, XLSX)",
                filetypes=[("Excel & CSV files", "*.csv *.xls *.xlsx"), ("All files", "*.*")]
            )
            file_paths = list(selected_files)
        except tk.TclError:
            print("Auto-detecting CSV, XLS, and XLSX files in the current folder...")
            for ext in ['*.csv', '*.xls', '*.xlsx']:
                file_paths.extend(glob.glob(ext))

    if file_paths:
        result_message = merge_files_to_json(file_paths[:10]) # Allow up to 10 files

        if 'google.colab' not in sys.modules:
            try:
                root = tk.Tk()
                root.withdraw()
                messagebox.showinfo("Merge Complete", result_message)
            except tk.TclError:
                pass
    else:
        print("No files to process. Exiting.")

Google Colab detected. Please upload your XLSX or CSV files below:


Saving Returns_Merged file (1).xlsx to Returns_Merged file (1).xlsx
Saving Shopify_Merged file (1).xlsx to Shopify_Merged file (1).xlsx
Saving Merged file 15April.xlsx to Merged file 15April.xlsx
Processing: Returns_Merged file (1).xlsx...
Processing: Shopify_Merged file (1).xlsx...
Processing: Merged file 15April.xlsx...
Successfully extracted data.
Total unique orders processed: 3769
Saved as: Bounty_Recon_Merged.json
